# videosplat — Experiments & Findings

**Free-viewpoint replay of dynamic scenes from multi-camera video, via 4D Gaussian Splatting.**

This notebook documents the experimental journey that built `videosplat`: a pipeline that takes casual multi-camera footage and produces an interactive 4D Gaussian Splat you can scrub through and fly around in a browser.

🔗 **Full code, README, and live demo:** [github.com/Ahmedn1/videosplat](https://github.com/Ahmedn1/videosplat)

---

We ran the pipeline on **five datasets** ranging from a synced 18-camera lab rig to a 3-camera casual phone capture with one camera literally pointing straight down at a piano keyboard. Each dataset taught us something different — and the most important lessons came from places we didn't expect.

| # | Dataset | Best metric | Key finding |
|---|---|---|---|
| 1 | 🏀 Basketball (12-cam synced benchmark) | — (validation) | Pipeline works end-to-end |
| 2 | ☕ Coffee Martini (N3V, 18-cam) | PSNR **28.08** (held-out-view) | Focal-units bug worth +3 PSNR; **coverage beats calibration accuracy** |
| 3 | 🥩 Flame Steak (N3V, STG path) | published-level | STG = fast-iteration path (~1h vs 24h COLMAP) |
| 4 | 🎹 Piano (3-cam casual, moving, unsynced) | PSNR **28.0** (held-out-time) | **Temporal density** is the dominant lever; novel-view overfits to camera locations |
| 5 | 💃 AIST Breakdance (9-cam synced ring) | PSNR **16.50** (held-out-view) | Novel-view collapse was an *optimization* bug, not geometry |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from PIL import Image
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',
    'grid.color': '#21262d',
    'grid.linewidth': 0.8,
    'axes.titlecolor': '#e6edf3',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'legend.fontsize': 10,
    'font.family': 'DejaVu Sans',
})

# ── asset path helper (works locally and on Kaggle) ──────────────────────────
ASSET_DIR = 'notebook_assets'

def show_img(fname, ax, title='', cmap=None):
    path = os.path.join(ASSET_DIR, fname)
    if os.path.exists(path):
        img = Image.open(path)
        ax.imshow(img, cmap=cmap)
    else:
        ax.text(0.5, 0.5, f'[{fname}\nnot found]', ha='center', va='center',
                transform=ax.transAxes, color='#6e7681', fontsize=9)
        ax.set_facecolor('#0d1117')
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=10, pad=4)

ACCENT = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']
print('Setup complete.')

---
## 1. Basketball — Phase 0: proving the pipeline

Before touching any custom capture, we needed to validate the full end-to-end pipeline on a known-good dataset. We chose the **Dynamic 3D Gaussians benchmark** — a multi-camera capture of a basketball/juggling performer from CMU Panoptic Studio, with calibration already provided.

**Why this?** No calibration uncertainty. No sync ambiguity. Clean textured scene. If the pipeline fails here, the failure is in the code.

**Outcome:** `extract → sync → calibrate → convert → train → bake → viewer` all ran end-to-end. Basketball became the regression dataset. The pipeline works.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), facecolor='#0d1117')
fig.suptitle('Basketball — 4D Gaussian Pipeline Validation (12-camera synced benchmark)',
             fontsize=13, color='#e6edf3', y=1.01)

steps = ['Extract frames', 'Calibrate poses', 'Convert format', 'Train 4DGS', 'Bake keyframes', 'Browser viewer']
status = [1, 1, 1, 1, 1, 1]

ax = axes[0]
bars = ax.barh(steps, status, color=ACCENT[1], alpha=0.85, height=0.6)
ax.set_xlim(0, 1.3)
ax.set_xlabel('Status')
ax.set_title('Pipeline stage completion', fontsize=11)
for bar in bars:
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height()/2,
            '✓', va='center', color=ACCENT[1], fontsize=13)
ax.set_xticks([])
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

ax2 = axes[1]
datasets = ['Basketball\n(benchmark)', 'Coffee\nMartini', 'Flame\nSteak', 'Piano', 'AIST\nBreakdance']
n_cams = [12, 18, 6, 3, 9]
colors = [ACCENT[1] if i == 0 else ACCENT[0] for i in range(5)]
ax2.bar(datasets, n_cams, color=colors, alpha=0.85, edgecolor='#21262d')
ax2.set_ylabel('Number of cameras')
ax2.set_title('Camera count per dataset', fontsize=11)
ax2.grid(axis='y', alpha=0.5)
ax2.set_ylim(0, 22)
for i, v in enumerate(n_cams):
    ax2.text(i, v + 0.3, str(v), ha='center', color='#e6edf3', fontsize=11, fontweight='bold')
legend_els = [mpatches.Patch(color=ACCENT[1], label='Phase-0 validation'),
              mpatches.Patch(color=ACCENT[0], label='Research datasets')]
ax2.legend(handles=legend_els, fontsize=9)

ax3 = axes[2]
algos = ['4DGaussians\n(4dgs)', 'SpaceTime\nGaussians (stg)', 'Gaussian-Flow\n(gflow)', '4D-Rotor\n(4d-rotor)']
iters = [14000, 20000, 30000, 30000]
calib_h = [24, 1, 24, 24]
x = np.arange(len(algos))
width = 0.35
ax3.bar(x - width/2, iters, width, label='Train iterations', color=ACCENT[0], alpha=0.85, edgecolor='#21262d')
ax3r = ax3.twinx()
ax3r.bar(x + width/2, calib_h, width, label='Calib time (h)', color=ACCENT[4], alpha=0.85, edgecolor='#21262d')
ax3.set_xticks(x)
ax3.set_xticklabels(algos, fontsize=8)
ax3.set_ylabel('Train iterations', color=ACCENT[0])
ax3r.set_ylabel('Calibration time (h)', color=ACCENT[4])
ax3.set_title('Algorithm comparison', fontsize=11)
ax3.tick_params(axis='y', colors=ACCENT[0])
ax3r.tick_params(axis='y', colors=ACCENT[4])
lines1 = [mpatches.Patch(color=ACCENT[0], label='Train iterations'),
          mpatches.Patch(color=ACCENT[4], label='Calibration time (h)')]
ax3.legend(handles=lines1, fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('notebook_assets/_chart_basketball.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Basketball section: pipeline validated end-to-end on a 12-camera synced benchmark.')

---
## 2. Coffee Martini — PSNR 13 → 28 (the quality bring-up)

The [Neural 3D Video dataset](https://github.com/facebookresearch/Neural_3D_Video) `coffee_martini` scene is an 18-camera synchronized GoPro capture of a coffee martini being poured, lit with hard studio lighting (glare, windows, difficult reflections).

We chose it because it has a **published held-out-view PSNR benchmark** (~28 dB) — a real target to chase. Starting PSNR was 13. The climb to 28 revealed two completely different lessons.

### The COLMAP calibration deep-dive (and why it didn't matter)

We initially assumed poor calibration was the PSNR bottleneck — the default approach was to run COLMAP on all 1800 frames (24h compute). Our hypothesis: fewer, better frames would calibrate more accurately.

We were right about calibration accuracy — but wrong about what it would do for PSNR.

In [ ]:
# ── Coffee Martini: calibration accuracy vs PSNR journey ─────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0d1117')
fig.suptitle('Coffee Martini (N3V, 18-cam) — PSNR 13 → 28 journey',
             fontsize=14, color='#e6edf3', y=1.02, fontweight='bold')

# ── left: calibration error vs frames/cam ────────────────────────────────────
ax1 = axes[0]
frames_per_cam = [1, 3, 5]
rot_err = [3.42, 1.77, 0.145]      # from results.tsv (deg)
baseline_err = 4.61                # 24h COLMAP on all 1800 frames

ax1.axhline(baseline_err, color=ACCENT[2], lw=1.5, ls='--', alpha=0.8,
            label=f'All 1800 frames (24h): {baseline_err:.2f}°')
ax1.plot(frames_per_cam, rot_err, 'o-', color=ACCENT[0], lw=2.5,
         markersize=8, label='Static-rig COLMAP')
for x, y in zip(frames_per_cam, rot_err):
    ax1.annotate(f'{y:.3f}°', (x, y), textcoords='offset points',
                 xytext=(8, -4), color=ACCENT[0], fontsize=9)
ax1.annotate('32× more accurate\n400× faster!', xy=(5, 0.145),
             xytext=(3.2, 1.5), fontsize=9, color=ACCENT[1],
             arrowprops=dict(arrowstyle='->', color=ACCENT[1], lw=1.5))
ax1.set_xlabel('Frames per camera used for calibration')
ax1.set_ylabel('Rotation error (degrees, lower=better)')
ax1.set_title('Calibration accuracy vs frame count')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.4)
ax1.set_xticks(frames_per_cam)

# ── middle: the key insight — calib accuracy ≠ PSNR ─────────────────────────
ax2 = axes[1]
experiments = ['Baseline\n(24h COLMAP)', 'Fast calib\n(5 frames/cam)', 'Focal\nunits fix', 'All cams +\ndownsample2']
psnr_vals = [12.76, 12.76, 15.28, 28.08]
calib_err_norm = [4.61, 0.145, 0.145, 0.145]

x = np.arange(len(experiments))
bar_colors = [ACCENT[2], ACCENT[0], ACCENT[4], ACCENT[1]]
bars = ax2.bar(x, psnr_vals, color=bar_colors, alpha=0.85, edgecolor='#21262d', width=0.6)
for bar, v in zip(bars, psnr_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{v:.1f}', ha='center', color='#e6edf3', fontsize=10, fontweight='bold')

# annotate the flat section
ax2.annotate('', xy=(1, 13.3), xytext=(0, 13.3),
             arrowprops=dict(arrowstyle='<->', color='#8b949e', lw=1.5))
ax2.text(0.5, 13.7, '32× better calib\n= 0 PSNR gain', ha='center',
         fontsize=8.5, color='#8b949e', style='italic')
ax2.annotate('', xy=(2, 16.0), xytext=(1, 16.0),
             arrowprops=dict(arrowstyle='->', color=ACCENT[4], lw=1.5))
ax2.text(1.5, 17.5, 'Focal-units\nbug fix\n+3 PSNR', ha='center', fontsize=8.5, color=ACCENT[4])
ax2.annotate('', xy=(3, 26.5), xytext=(2, 26.5),
             arrowprops=dict(arrowstyle='->', color=ACCENT[1], lw=1.5))
ax2.text(2.5, 24.5, 'Coverage\n+13 PSNR!', ha='center', fontsize=8.5, color=ACCENT[1], fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels(experiments, fontsize=9)
ax2.set_ylabel('Test PSNR (dB)')
ax2.set_title('Test PSNR by experiment step')
ax2.set_ylim(8, 31)
ax2.axhline(28.08, color=ACCENT[1], ls='--', lw=1, alpha=0.4)
ax2.grid(axis='y', alpha=0.4)

# ── right: training curve of the winning run ─────────────────────────────────
ax3 = axes[2]
# approximate training curve from logged values
iters_log = [3000, 7000, 14000]
psnr_maxq  = [27.67, 27.76, 28.08]   # exp6 — all-cams + downsample2 (the winner)
psnr_focal = [None, None, 15.28]      # exp5 — focal fix only
psnr_bad   = [13.24, 12.69, 12.76]   # psnr-1 — baseline calib

ax3.plot(iters_log, psnr_bad, 'o--', color=ACCENT[2], lw=2, markersize=7, label='Baseline (13→13)', alpha=0.8)
ax3.plot(iters_log, psnr_maxq, 'o-', color=ACCENT[1], lw=2.5, markersize=8, label='All cams + downsample=2 (winner)')
ax3.axhline(28.08, color=ACCENT[1], ls=':', lw=1, alpha=0.5)
ax3.text(14200, 28.2, '28.08 dB', color=ACCENT[1], fontsize=9)
ax3.set_xlabel('Training iterations')
ax3.set_ylabel('Test PSNR (dB)')
ax3.set_title('Training curve comparison')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.4)
ax3.set_xlim(1000, 16000)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_coffee_martini.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

### The focal-units bug

The 4DGaussians `dynerf` reader hardcodes N3V's native **2704px** width and assumes `poses_bounds` focals are stored in those pixels. We were extracting frames at 1280px — so every camera was rendering with a field of view ~**2.1×** too wide. Everything was subtly blurred because the model was trying to explain a wider scene than existed.

The fix was one line in `convert.py`: rescale the stored focal to the 2704px reference width. **+3 dB PSNR from a unit bug.**

### The real ceiling: camera coverage

After the focal fix, test PSNR was ~15. We assumed more training might help. It didn't.

What helped was using **all 17 training cameras** (vs subsets) and **`--downsample 2`** (4× more pixels per training image). The key intuition: held-out-view quality is gated by how well the training cameras *surround* the held-out one. Add cameras → **+12 dB**. No new model architecture, no tuning.

In [ ]:
# ── Coffee Martini: rendered time frames at t=0, 0.5, 1.0 ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='#0d1117')
fig.suptitle('Coffee Martini — Three moments in time (t=0, 0.5, 1.0)  |  Held-out-view PSNR 28.08 dB',
             fontsize=12, color='#e6edf3', y=1.01)

show_img('cm_t0.png', axes[0], 't = 0.0  (start of pour)')
show_img('cm_t1.png', axes[1], 't = 0.5  (mid-pour)')
show_img('cm_t2.png', axes[2], 't = 1.0  (end of pour)')

plt.tight_layout()
plt.savefig('notebook_assets/_chart_cm_renders.png', dpi=100, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Frames rendered from the held-out camera viewpoint at three time steps.')

**Key findings from Coffee Martini:**
1. **Calibration accuracy ≠ PSNR.** Making COLMAP 32× more accurate (and 400× faster) did *not* move test PSNR. We invalidated our core proxy metric mid-study.
2. **Focal units matter.** A single-line unit bug worth 3 dB that persists silently — always sanity-check your focal length scale.
3. **Coverage is the dominant novel-view lever.** Training camera distribution matters more than any model hyperparameter.

---
## 3. Flame Steak — SpacetimeGaussians: the fast-iteration path

Another N3V scene, but reconstructed with **SpacetimeGaussians (STG)** instead of 4DGaussians. STG uses per-Gaussian polynomial motion instead of a deformation network — it's faster to render and uses less VRAM.

The key operational difference: STG calibrates with **~50 small per-frame COLMAPs (~1h total)** versus the single 24h exhaustive COLMAP that 4dgs needs on N3V. Since calibration accuracy turned out *not* to be the PSNR bottleneck (see above), STG is the right tool for **fast iteration**.

In [ ]:
# ── Flame Steak: STG vs 4DGS trade-off visualization ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0d1117')
fig.suptitle('Flame Steak — SpacetimeGaussians (STG): Speed vs Quality trade-off',
             fontsize=13, color='#e6edf3', y=1.02, fontweight='bold')

# ── left: calibration time comparison ────────────────────────────────────────
ax1 = axes[0]
methods = ['4DGaussians\n(4dgs)\nsingle COLMAP', 'SpacetimeGaussians\n(stg)\nper-frame COLMAPs']
calib_times = [24, 1]
bar_colors = [ACCENT[0], ACCENT[1]]
bars = ax1.bar(methods, calib_times, color=bar_colors, alpha=0.85, edgecolor='#21262d', width=0.5)
for bar, v in zip(bars, calib_times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{v}h', ha='center', color='#e6edf3', fontsize=14, fontweight='bold')
ax1.set_ylabel('Calibration time (hours)')
ax1.set_title('Calibration cost: 4dgs vs STG')
ax1.set_ylim(0, 28)
ax1.grid(axis='y', alpha=0.4)
ax1.annotate('24× faster\n(same PSNR impact)', xy=(1, 1.2), xytext=(0.5, 15),
             fontsize=10, color=ACCENT[1], ha='center',
             arrowprops=dict(arrowstyle='->', color=ACCENT[1], lw=1.5))

# ── right: when to use what ───────────────────────────────────────────────────
ax2 = axes[1]
ax2.axis('off')

table_data = [
    ['Property', '4dgs', 'stg'],
    ['Default iterations', '14 000', '20 000'],
    ['Calibration time', '~24h', '~1h'],
    ['VRAM (training)', 'high', 'lower'],
    ['Best for', 'max fidelity', 'fast iteration'],
    ['Moving/unsynced cams', '✓ (nerfies mode)', '—'],
    ['Real-time render', 'possible', '✓ native'],
]

col_widths = [0.3, 0.25, 0.25]
col_starts = [0.0, 0.35, 0.62]
row_h = 0.11
row_start = 0.9

for r, row in enumerate(table_data):
    y = row_start - r * row_h
    is_header = (r == 0)
    for c, cell in enumerate(row):
        x = col_starts[c]
        fc = '#21262d' if is_header else ('#161b22' if r % 2 == 0 else '#0d1117')
        rect = mpatches.FancyBboxPatch((x + 0.01, y - row_h + 0.01),
                                        col_widths[c] - 0.02, row_h - 0.015,
                                        boxstyle='round,pad=0.01', facecolor=fc,
                                        edgecolor='#30363d', lw=0.8)
        ax2.add_patch(rect)
        color = '#e6edf3' if is_header else (
            ACCENT[1] if '✓' in cell else ACCENT[2] if '—' == cell else '#c9d1d9')
        ax2.text(x + col_widths[c]/2, y - row_h/2, cell, ha='center', va='center',
                 fontsize=9.5 if not is_header else 10,
                 fontweight='bold' if is_header else 'normal', color=color)

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_title('4dgs vs STG: when to use which', fontsize=11)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_flame_steak.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

**Key finding:** STG is the right tool for fast iteration. Since calibration accuracy doesn't predict reconstruction quality (as proven on Coffee Martini), spending 24h on calibration when 1h gives the same result is waste. Use STG to prototype; use 4dgs when you want maximum fidelity.

---
## 4. Piano — Casual 3-Camera Capture (the hard problem)

This is where things get genuinely difficult. The piano capture is **deliberately adversarial**:
- **3 phones**, not a calibrated rig
- **Cam0**: side view (1280×720)
- **Cam1**: **top-down** (1080×1920, pointing straight down at the keys)
- **Cam2**: other side (1024×576) — and **physically moving** in the second half of the clip
- Different start times and lengths
- No ground truth poses

This is what `videosplat casual` was built for.

In [ ]:
# ── Piano: the three training camera views ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), facecolor='#0d1117')
fig.suptitle('Piano — Three heterogeneous, unsynced cameras (the training views)',
             fontsize=13, color='#e6edf3', y=1.01, fontweight='bold')

show_img('piano_cam0.png', axes[0], 'Cam 0 — side view (1280×720)\nstatic')
show_img('piano_cam1.png', axes[1], 'Cam 1 — TOP-DOWN (1080×1920)\nstatic (widest baseline)')
show_img('piano_cam2.png', axes[2], 'Cam 2 — other side (1024×576)\nMOVING in 2nd half')

for ax in axes:
    ax.set_facecolor('#0d1117')
plt.tight_layout()
plt.savefig('notebook_assets/_chart_piano_cams.png', dpi=100, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('SIFT/COLMAP links 0 of 8 images for Cam1. MASt3R links all 3 cameras.')

### Why COLMAP failed — and why MASt3R worked

COLMAP uses SIFT features. SIFT fundamentally cannot register cam1 (top-down, looking straight at piano keys) to the side views — the geometry is too different and the piano keys are too repetitive. The calibration output: **0/8 frames registered for cam1**.

MASt3R uses learned dense matching instead of hand-crafted keypoints. It links all three cameras into a coherent joint point cloud (206k dense points).

### The temporal density lever

In [ ]:
# ── Piano: temporal density is THE lever ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0d1117')
fig.suptitle('Piano — Temporal Density: the dominant quality lever',
             fontsize=13, color='#e6edf3', y=1.02, fontweight='bold')

# ── left: PSNR vs n_timesteps ─────────────────────────────────────────────────
ax1 = axes[0]
n_ts   = [20,   150,  300,  600]
psnr_ts = [16.26, 24.58, 27.73, 28.03]

ax1.fill_between(n_ts, psnr_ts, alpha=0.15, color=ACCENT[0])
ax1.plot(n_ts, psnr_ts, 'o-', color=ACCENT[0], lw=2.5, markersize=10, zorder=5)
for x, y in zip(n_ts, psnr_ts):
    ax1.annotate(f'{y:.1f} dB', (x, y), textcoords='offset points',
                 xytext=(8, 6), fontsize=10, color=ACCENT[0], fontweight='bold')

# annotate the deformation-hurts region
ax1.annotate('Deformation\nhurts at 20 ts\n(can\'t interpolate)', xy=(20, 16.26),
             xytext=(60, 17.5), fontsize=8.5, color=ACCENT[2],
             arrowprops=dict(arrowstyle='->', color=ACCENT[2], lw=1.2))
ax1.annotate('Sweet spot\n≈ 300 ts', xy=(300, 27.73),
             xytext=(200, 24.5), fontsize=9, color=ACCENT[1],
             arrowprops=dict(arrowstyle='->', color=ACCENT[1], lw=1.5))
ax1.axvline(300, color=ACCENT[1], ls='--', lw=1.2, alpha=0.5)
ax1.axhline(28.08, color='#8b949e', ls=':', lw=1, alpha=0.6)
ax1.text(620, 28.2, 'Coffee Martini\nbest (28.08)', color='#8b949e', fontsize=8)

ax1.set_xscale('log')
ax1.set_xticks([20, 150, 300, 600])
ax1.set_xticklabels(['20', '150', '300', '600'])
ax1.set_xlabel('Number of temporal timesteps (n_time)')
ax1.set_ylabel('Held-out-TIME PSNR (dB)')
ax1.set_title('PSNR vs temporal density (Piano)')
ax1.grid(alpha=0.4)
ax1.set_ylim(14, 30)

# ── right: why more iterations / resolution doesn't help ─────────────────────
ax2 = axes[1]
params = ['14k iters\n(default)', '30k iters', '1024×576\n(default res)', '1600×900\n(higher res)']
psnr_p = [28.03, 24.31, 28.03, 23.88]
colors_p = [ACCENT[1], ACCENT[2], ACCENT[1], ACCENT[2]]
bars = ax2.bar(params, psnr_p, color=colors_p, alpha=0.85, edgecolor='#21262d', width=0.55)
for bar, v, c in zip(bars, psnr_p, colors_p):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{v:.1f}', ha='center', color=c, fontsize=11, fontweight='bold')
ax2.axhline(28.03, color=ACCENT[1], ls='--', lw=1.2, alpha=0.5)
ax2.text(3.6, 28.25, 'best (28.03)', color=ACCENT[1], fontsize=8)
ax2.set_ylabel('Held-out-TIME PSNR (dB)')
ax2.set_title('More compute does NOT help sparse captures')
ax2.set_ylim(18, 31)
ax2.grid(axis='y', alpha=0.4)
legend_els = [mpatches.Patch(color=ACCENT[1], label='Best / recommended'),
              mpatches.Patch(color=ACCENT[2], label='Regressed')]
ax2.legend(handles=legend_els, fontsize=9)
ax2.text(0.5, -0.18, '→ Sparse-view captures are not compute-limited. Spend the budget on temporal density and coverage.',
         ha='center', transform=ax2.transAxes, fontsize=9, color='#8b949e', style='italic')

plt.tight_layout()
plt.savefig('notebook_assets/_chart_piano_density.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

### Person masking — downweight, don't blackout

The pianist's hands move constantly over the keyboard. In the MASt3R global pose alignment, those moving hands vote on the *static* camera poses — and they're wrong votes (the hand isn't a fixed landmark).

The naive fix is to **black out the person** (set those pixels to 0). It doesn't work: blackout also removes the keyboard at the hands — the exact static structure the calibration needs.

The fix is **confidence-downweighting**: detect people with Mask R-CNN, then lower the matching confidence weight for those pixels — `conf ← 1 + (conf − 1) × (1 − s)` where `s` is downweight strength. The full image stays intact; the person just contributes less to pose.

In [ ]:
# ── Piano: masking comparison chart + viz ────────────────────────────────────
fig = plt.figure(figsize=(15, 5), facecolor='#0d1117')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.3)
fig.suptitle('Piano — Person masking: downweight vs blackout vs unmasked',
             fontsize=13, color='#e6edf3', y=1.02, fontweight='bold')

# ── left: PSNR comparison bar ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
methods = ['Black-out\n(naive mask)', 'No masking\n(unmasked)', 'Conf-downweight\n(ours)']
psnrs = [18.98, 23.31, 23.65]
mc = [ACCENT[2], ACCENT[4], ACCENT[1]]
bars = ax1.bar(methods, psnrs, color=mc, alpha=0.85, edgecolor='#21262d', width=0.55)
for bar, v, c in zip(bars, psnrs, mc):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{v:.2f}', ha='center', color=c, fontsize=11, fontweight='bold')
ax1.set_ylabel('Frontier held-out-time PSNR (dB)')
ax1.set_title('Masking strategy vs PSNR\n(cam2 frontier variant)', fontsize=11)
ax1.set_ylim(15, 26)
ax1.grid(axis='y', alpha=0.4)

ax1.annotate('Removes keyboard\nat player\'s hands!', xy=(0, 19.3), xytext=(-0.3, 22),
             fontsize=8, color=ACCENT[2],
             arrowprops=dict(arrowstyle='->', color=ACCENT[2], lw=1.2))
ax1.annotate('Best!', xy=(2, 23.85), xytext=(1.8, 25),
             fontsize=9, color=ACCENT[1], fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=ACCENT[1], lw=1.5))

# ── middle: masking visualization ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
show_img('piano_maskviz.jpg', ax2, 'Person mask (Mask R-CNN)\nDownweight region shown in red')

# ── right: the formula explanation ───────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.axis('off')
ax3.set_facecolor('#0d1117')

formula_text = (
    'Confidence-downweighting formula:\n\n'
    '  conf_person = 1 + (conf − 1) × (1 − s)\n\n'
    '  s = --mask-downweight  ∈ [0, 1]\n'
    '  s=0 → no change\n'
    '  s=1 → person pixels have zero\n'
    '         influence on pose\n\n'
    'Key insight:\n'
    '  The full image is preserved.\n'
    '  Static structure (keyboard)\n'
    '  next to the hands STILL\n'
    '  contributes to calibration.\n\n'
    'Blackout removes static features\n'
    'adjacent to the person — exactly\n'
    'the keyboard at the hands that\n'
    'the solver needs most.'
)
ax3.text(0.05, 0.95, formula_text, transform=ax3.transAxes,
         fontsize=9.5, color='#c9d1d9', va='top', fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#161b22', edgecolor='#30363d'))
ax3.set_title('How confidence-downweighting works', fontsize=11)

plt.savefig('notebook_assets/_chart_piano_masking.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

### Novel-time vs novel-view — the overfit we almost missed

With held-out-time PSNR at 28.0, the piano model looked like a success. We had tested *temporal* generalization — holding out entire time segments the model hadn't seen.

What we hadn't tested: **held-out-VIEW** — rendering from angles the cameras never saw.

The result was striking.

In [ ]:
# ── Piano: held-out-time vs held-out-view proof ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='#0d1117')
fig.suptitle('Piano — Novel-Time (28.0 dB) vs Novel-View (floater collapse)',
             fontsize=13, color='#e6edf3', y=1.01, fontweight='bold')

show_img('piano_proof_near.png', axes[0],
         'From cam0 pose (seen during training)\nHeld-out-TIME PSNR: 28.0 dB  ✓ Sharp')
show_img('piano_proof_far.png', axes[1],
         'From 2.5× scene radius (novel angle)\nHeld-out-VIEW: explodes into floaters  ✗')

# annotate
axes[0].set_title('From cam0 pose (seen during training)\nHeld-out-TIME PSNR: 28.0 dB  ✓ Sharp',
                  color=ACCENT[1], fontsize=10)
axes[1].set_title('From 2.5× scene radius (novel angle)\nHeld-out-VIEW: explodes into floaters  ✗',
                  color=ACCENT[2], fontsize=10)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_piano_overfit.png', dpi=100, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('A 3-camera capture only constrains geometry where cameras overlap.')
print('The model overfits to the training camera locations — not a failure of training, a geometric fact.')

A 3-camera capture only constrains geometry where the cameras overlap. The 4DGS model correctly memorizes a consistent appearance from those three viewpoints — but outside that cone, there's no geometric constraint, so it fills empty space with floaters.

**This is not a training failure.** It's a geometric consequence of sparse coverage. The usable free-viewpoint product is a gentle micro-orbit staying inside the captured viewing cone — not a full fly-around.

**The lesson:** always test held-out-VIEW if you claim free-viewpoint. A great held-out-time score says nothing about novel views.

### The walkthrough result

In [ ]:
# ── Piano: walkthrough frames ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='#0d1117')
fig.suptitle('Piano — Walkthrough orbit (14% scene-radius orbit inside the captured cone)',
             fontsize=12, color='#e6edf3', y=1.01)

show_img('piano_walk_3s.png',  axes[0], 't = 3s  (opening)')
show_img('piano_walk_15s.png', axes[1], 't = 15s (mid-performance)')
show_img('piano_walk_27s.png', axes[2], 't = 27s (finale)')

plt.tight_layout()
plt.savefig('notebook_assets/_chart_piano_walkthrough.png', dpi=100, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

---
## 5. AIST Breakdance — 9-camera synced ring (novel-view: 15.13 → 16.50)

The [AIST Dance Video Database](https://aistdancedb.ongaaccel.jp/) `breakdance_ch01` scene: a breakdancer in a white cyclorama, 9 hardware-synced 1080p cameras in a 360° ring. Camera calibration comes from [AIST++](https://google.github.io/aistplusplus_dataset/).

This was the first dataset where we could do a real **held-out-VIEW** evaluation: hold out 2 of 9 cameras, train on 7, measure PSNR on the novel views.

We expected the white textureless cyclorama to be a MASt3R calibration blocker. It wasn't — the synced dancer provides strong cross-camera correspondences even in a blank white studio.

The real problem was elsewhere.

In [ ]:
# ── AIST: the 360° camera ring ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(17, 4), facecolor='#0d1117')
fig.suptitle('AIST Breakdance — 9 cameras in a 360° ring (5 shown; cameras 2 and 6 held out for novel-view eval)',
             fontsize=12, color='#e6edf3', y=1.01)

cams = ['aist_cam0.png', 'aist_cam2.png', 'aist_cam4.png', 'aist_cam6.png', 'aist_cam8.png']
labels = ['Cam 0 (train)', 'Cam 2 (HELD-OUT)', 'Cam 4 (train)', 'Cam 6 (HELD-OUT)', 'Cam 8 (train)']
holdout = [False, True, False, True, False]

for ax, cam, label, is_ho in zip(axes, cams, labels, holdout):
    show_img(cam, ax, label)
    color = ACCENT[2] if is_ho else ACCENT[1]
    ax.set_title(label, color=color, fontsize=9, pad=3)
    if is_ho:
        for spine in ['top', 'bottom', 'left', 'right']:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_color(ACCENT[2])
            ax.spines[spine].set_linewidth(2)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_aist_cams.png', dpi=100, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

### The novel-view collapse — an optimization bug

The first AIST reconstruction (HyperNeRF defaults) had held-out-view PSNR 15.13. But more worrying: the test PSNR **degraded across training** — 15.9 at 3k iterations, 15.1 at 14k. The model was getting *worse* at the held-out views as it trained longer.

This is the signature of floater collapse: the model is filling empty space with semi-transparent Gaussians that happen to satisfy the 7 training cameras, and these floaters actively hurt the held-out angles.

The root cause: **HyperNeRF's default config sets `opacity_reset_interval=300000`** — which in a 14k-iteration run means "never reset." Standard 3DGS opacity reset is the main mechanism for removing floaters. It was completely disabled.

In [ ]:
# ── AIST: experiment progression ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0d1117')
fig.suptitle('AIST Breakdance — Held-out-VIEW PSNR: 15.13 → 16.50 (optimization bug fix)',
             fontsize=13, color='#e6edf3', y=1.02, fontweight='bold')

# ── left: experiment steps ────────────────────────────────────────────────────
ax1 = axes[0]
exp_labels = [
    'Exp 2\nHyperNeRF\ndefaults',
    'Exp 3\n+opacity_reset\n+dssim=0.2',
    'Exp 4\n+opacity_reset\n(no dssim)',
    'Exp 5\n+reduced\ndensification',
    'Exp 6\ndensify grad\n(plateau)',
]
psnr_aist = [15.13, 16.17, 16.13, 16.43, 16.50]
highlight = [False, True, False, True, True]
bar_colors = [ACCENT[2] if i == 0 else (ACCENT[1] if h else ACCENT[0])
              for i, h in enumerate(highlight)]

bars = ax1.bar(range(5), psnr_aist, color=bar_colors, alpha=0.85, edgecolor='#21262d', width=0.6)
for i, (bar, v) in enumerate(zip(bars, psnr_aist)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f'{v:.2f}', ha='center', color='#e6edf3', fontsize=10, fontweight='bold')

ax1.annotate('opacity_reset\nrestored → +1.04!', xy=(1, 16.25), xytext=(2.5, 15.5),
             fontsize=9, color=ACCENT[1], ha='center',
             arrowprops=dict(arrowstyle='->', color=ACCENT[1], lw=1.5))
ax1.annotate('dssim: +0.04\n= noise only\n(3× slower!)', xy=(2, 16.2), xytext=(2.8, 16.4),
             fontsize=8, color=ACCENT[4],
             arrowprops=dict(arrowstyle='->', color=ACCENT[4], lw=1))

ax1.set_xticks(range(5))
ax1.set_xticklabels(exp_labels, fontsize=8)
ax1.set_ylabel('Held-out-VIEW PSNR (dB)')
ax1.set_title('Experiment progression')
ax1.set_ylim(14.5, 17)
ax1.grid(axis='y', alpha=0.4)
legend_els = [mpatches.Patch(color=ACCENT[2], label='Baseline (HyperNeRF defaults)'),
              mpatches.Patch(color=ACCENT[0], label='Keep'),
              mpatches.Patch(color=ACCENT[1], label='Best result')]
ax1.legend(handles=legend_els, fontsize=9)

# ── right: training curve showing degradation vs fix ─────────────────────────
ax2 = axes[1]
iters_a = [3000, 7000, 14000]
# approximate from logs
psnr_default = [15.92, 15.42, 15.13]  # exp2: degrades
psnr_fixed   = [16.67, 16.23, 16.17]  # exp3: flat/stable
psnr_best    = [16.72, 16.55, 16.50]  # exp6: best

ax2.plot(iters_a, psnr_default, 'o--', color=ACCENT[2], lw=2, markersize=8,
         label='HyperNeRF defaults (opacity_reset=NEVER)')
ax2.plot(iters_a, psnr_fixed,   'o-',  color=ACCENT[0], lw=2, markersize=8,
         label='opacity_reset=3000')
ax2.plot(iters_a, psnr_best,    's-',  color=ACCENT[1], lw=2.5, markersize=9,
         label='opacity_reset + reduced densification (best)')

ax2.annotate('Test PSNR\nDEGRADES\n(floater collapse)', xy=(14000, 15.13),
             xytext=(9000, 14.9), fontsize=8.5, color=ACCENT[2],
             arrowprops=dict(arrowstyle='->', color=ACCENT[2], lw=1.2))
ax2.set_xlabel('Training iterations')
ax2.set_ylabel('Held-out-VIEW PSNR (dB)')
ax2.set_title('Training curves: opacity reset is critical')
ax2.legend(fontsize=8.5)
ax2.grid(alpha=0.4)
ax2.set_ylim(14.5, 17.2)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_aist_experiments.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# ── AIST: held-out-view before vs after ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8), facecolor='#0d1117')
fig.suptitle('AIST — Held-out-view: baseline (floaters) vs best (dancer resolved)',
             fontsize=12, color='#e6edf3', y=1.01, fontweight='bold')

show_img('aist_exp2_cmp_a.png', axes[0][0], 'Baseline (exp 2) — frame 40 — PSNR 15.13')
show_img('aist_exp2_cmp_b.png', axes[0][1], 'Baseline (exp 2) — frame 140')
show_img('aist_exp6_cmp_a.png', axes[1][0], 'Best (exp 6) — frame 40 — PSNR 16.50')
show_img('aist_exp6_cmp_b.png', axes[1][1], 'Best (exp 6) — frame 140')

axes[0][0].set_title('Baseline — floaters fill held-out view', color=ACCENT[2], fontsize=10)
axes[0][1].set_title('Baseline — dancer barely visible', color=ACCENT[2], fontsize=10)
axes[1][0].set_title('Best — dancer clearly resolved', color=ACCENT[1], fontsize=10)
axes[1][1].set_title('Best — consistent across frames', color=ACCENT[1], fontsize=10)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_aist_compare.png', dpi=100, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Note: comparison images show the rendered held-out camera view (left or right half) vs ground truth (other half).')

**Key findings from AIST:**
1. **MASt3R calibrates white textureless studios fine.** The synced dancer is the shared feature — 9/9 cameras registered, clean 360° ring (radius CV 11%, planarity 4%).
2. **The novel-view collapse was an optimization bug, not scene geometry.** Restoring standard 3DGS `opacity_reset_interval=3000` was +1.04 PSNR and removed the degradation signature.
3. **`dssim` was pure cost** (+0.04 = noise, ~3× slower). Dropped from defaults.
4. **Ceiling ~16.5 is the textureless background.** No features on white walls → floaters persist in unobserved volume. The dancer reconstructs well; the empty studio caps the metric.

**This fix is now the default** for the `casual`/nerfies path: `opacity_reset_interval=3000` unless explicitly overridden.

---
## 6. Cross-cutting lessons

Synthesizing across all five datasets.

In [ ]:
# ── Cross-cutting: overall PSNR summary ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='#0d1117')
fig.suptitle('Cross-dataset Summary — PSNR improvements across five datasets',
             fontsize=13, color='#e6edf3', y=1.02, fontweight='bold')

# ── left: start vs best PSNR ─────────────────────────────────────────────────
ax1 = axes[0]
datasets_n = ['Coffee Martini\n(held-out-view)', 'Piano\n(held-out-time)', 'AIST Breakdance\n(held-out-view)']
psnr_start = [12.76, 16.26, 15.13]
psnr_best_n = [28.08, 28.03, 16.50]

x = np.arange(len(datasets_n))
width = 0.35

bars_s = ax1.bar(x - width/2, psnr_start, width, label='Starting PSNR', color=ACCENT[2], alpha=0.8, edgecolor='#21262d')
bars_b = ax1.bar(x + width/2, psnr_best_n, width, label='Best achieved PSNR', color=ACCENT[1], alpha=0.85, edgecolor='#21262d')

for bar, v in zip(bars_s, psnr_start):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{v:.1f}', ha='center', color=ACCENT[2], fontsize=9, fontweight='bold')
for bar, v in zip(bars_b, psnr_best_n):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{v:.1f}', ha='center', color=ACCENT[1], fontsize=9, fontweight='bold')

# gain arrows
for i, (s, b) in enumerate(zip(psnr_start, psnr_best_n)):
    gain = b - s
    ax1.annotate('', xy=(i + width/2, b + 0.8), xytext=(i - width/2, s + 0.8),
                 arrowprops=dict(arrowstyle='->', color='#e6edf3', lw=1.5,
                                 connectionstyle='arc3,rad=0.3'))
    ax1.text(i + 0.0, max(s, b) + 2.2, f'+{gain:.1f} dB', ha='center',
             color='#e6edf3', fontsize=9, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(datasets_n, fontsize=9)
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('Start vs. best PSNR')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.4)
ax1.set_ylim(8, 33)

# ── right: lessons table ──────────────────────────────────────────────────────
ax2 = axes[1]
ax2.axis('off')

lessons = [
    ('Calibration accuracy ≠ PSNR',      '32× better calib = 0 dB gain (Coffee Martini)',   ACCENT[4]),
    ('Coverage beats calibration',        'All 17 cams vs subset = +13 dB (Coffee Martini)', ACCENT[1]),
    ('Temporal density is the lever',     '20→600 timesteps: +12 dB (Piano)',                ACCENT[1]),
    ('Restore opacity reset',             '+1 dB, removes degradation signature (AIST)',      ACCENT[1]),
    ('Downweight, don\'t blackout',       'Conf-downweight +4.7 dB vs naive mask (Piano)',   ACCENT[0]),
    ('Novel-time ≠ novel-view',           '28 dB → floater collapse at novel angles (Piano)',ACCENT[2]),
    ('Sparse views: not compute-limited', '30k iters & higher res both REGRESSED (Piano)',   ACCENT[4]),
    ('MASt3R for hard geometry',          'Top-down cam, textureless studio: all work',       ACCENT[0]),
]

y_start = 0.96
row_h = 0.11
for i, (title, detail, col) in enumerate(lessons):
    y = y_start - i * row_h
    ax2.add_patch(mpatches.FancyBboxPatch(
        (0.01, y - row_h + 0.01), 0.98, row_h - 0.015,
        boxstyle='round,pad=0.01', facecolor='#161b22', edgecolor='#21262d', lw=0.8))
    ax2.add_patch(mpatches.FancyBboxPatch(
        (0.01, y - row_h + 0.01), 0.005, row_h - 0.015,
        boxstyle='square,pad=0', facecolor=col, edgecolor='none'))
    ax2.text(0.03, y - row_h/2, title, va='center', fontsize=9,
             color='#e6edf3', fontweight='bold')
    ax2.text(0.03, y - row_h/2 - 0.035, detail, va='center', fontsize=7.5, color='#8b949e')

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_title('Key lessons (across all five datasets)', fontsize=11)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_crosscutting.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# ── The big picture: what each dataset's main contribution was ───────────────
fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0d1117')
fig.suptitle('The research arc — what each dataset contributed',
             fontsize=13, color='#e6edf3', y=1.01, fontweight='bold')

phases = [
    (0.0, '🏀 Basketball\nPhase-0 validation\n(pipeline works)', ACCENT[0]),
    (0.25, '☕ Coffee Martini\nCalibration deep-dive\nFocal bug + coverage lesson', ACCENT[4]),
    (0.50, '🥩 Flame Steak\nFast iteration path\nSTG: 24h → 1h calib', ACCENT[3]),
    (0.75, '🎹 Piano\nCasual capture\nMASt3R + density + overfit', ACCENT[5]),
    (1.00, '💃 AIST Breakdance\n360° ring\nOpacity_reset + novel-view', ACCENT[1]),
]

# timeline line
ax.axhline(0.5, color='#30363d', lw=2, zorder=1)

for i, (x, label, col) in enumerate(phases):
    ax.scatter([x], [0.5], s=300, color=col, zorder=5, edgecolors='#e6edf3', lw=1.5)
    y_off = 0.72 if i % 2 == 0 else 0.28
    ax.annotate(label, (x, 0.5),
                xytext=(x, y_off),
                ha='center', va='center' if i % 2 == 0 else 'center',
                fontsize=9.5, color='#e6edf3',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22', edgecolor=col, lw=1.5),
                arrowprops=dict(arrowstyle='-', color='#30363d', lw=1))

# key lessons alongside timeline
key_milestones = [
    (0.25, -0.02, '→ Calibration\naccuracy\n≠ PSNR', ACCENT[4]),
    (0.50,  0.02, '→ Fast calib\n(STG)', ACCENT[3]),
    (0.75, -0.02, '→ Temporal\ndensity\n(dominant lever)', ACCENT[5]),
    (1.00,  0.02, '→ opacity_reset\nnow default', ACCENT[1]),
]

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()
plt.savefig('notebook_assets/_chart_research_arc.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

---
## 7. Calibration: COLMAP vs MASt3R

We support two calibration backends, tuned for different capture types.

In [ ]:
# ── Calibration decision chart ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6), facecolor='#0d1117')
fig.suptitle('COLMAP vs MASt3R — which calibration backend for your capture?',
             fontsize=13, color='#e6edf3', y=1.01, fontweight='bold')
ax.axis('off')

criteria = [
    'Textured scene (lots of SIFT features)',
    'Hardware-synced rig',
    'Wide-baseline / unusual angles',
    'Top-down / low-texture cameras',
    'Unsynced / casual phones',
    'Per-camera intrinsics needed',
    'Calibration speed (cost)',
    'Handles moving cameras',
]
colmap_scores = ['✓ Great', '✓ Great', '✗ May fail', '✗ Fails', '~ Okay', '✗ Shared only', '★ Fast', '✗ Static only']
mast3r_scores = ['✓ Great', '✓ Great', '✓ Great', '✓ Great', '✓ Great', '✓ Per-camera', '~ Slower', '✓ Per-frame poses']

col_starts = [0.02, 0.5, 0.75]
col_w = [0.46, 0.23, 0.23]
headers = ['Criterion', 'COLMAP (SIFT)', 'MASt3R (dense matching)']
header_colors = ['#e6edf3', ACCENT[0], ACCENT[3]]

row_h = 0.085
y0 = 0.97

# header
for c, (hdr, hcol) in enumerate(zip(headers, header_colors)):
    ax.add_patch(mpatches.FancyBboxPatch(
        (col_starts[c] + 0.005, y0 - row_h + 0.005), col_w[c] - 0.01, row_h - 0.01,
        boxstyle='round,pad=0.01', facecolor='#21262d', edgecolor='#30363d', lw=1))
    ax.text(col_starts[c] + col_w[c]/2, y0 - row_h/2, hdr, va='center', ha='center',
            fontsize=10, fontweight='bold', color=hcol)

for i, (crit, cs, ms) in enumerate(zip(criteria, colmap_scores, mast3r_scores)):
    y = y0 - (i + 1) * row_h
    fc = '#161b22' if i % 2 == 0 else '#0d1117'

    ax.add_patch(mpatches.FancyBboxPatch(
        (col_starts[0] + 0.005, y - row_h + 0.005), col_w[0] - 0.01, row_h - 0.01,
        boxstyle='round,pad=0.01', facecolor=fc, edgecolor='#21262d', lw=0.5))
    ax.text(col_starts[0] + col_w[0]/2, y - row_h/2, crit, va='center', ha='center',
            fontsize=9, color='#c9d1d9')

    for val, ci in [(cs, 1), (ms, 2)]:
        vc = ACCENT[1] if val.startswith('✓') else (ACCENT[2] if val.startswith('✗') else ACCENT[4])
        ax.add_patch(mpatches.FancyBboxPatch(
            (col_starts[ci] + 0.005, y - row_h + 0.005), col_w[ci] - 0.01, row_h - 0.01,
            boxstyle='round,pad=0.01', facecolor=fc, edgecolor='#21262d', lw=0.5))
        ax.text(col_starts[ci] + col_w[ci]/2, y - row_h/2, val, va='center', ha='center',
                fontsize=8.5, color=vc, fontweight='bold')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('notebook_assets/_chart_calibration.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

---
## 8. Conclusion

### What we built

`videosplat` is a complete pipeline for turning multi-camera video into interactive 4D Gaussian splats. It handles the full spectrum from calibrated synced lab rigs to casual unsynced phone captures.

### The five things that actually moved the needle

1. **Camera coverage** over calibration accuracy (Coffee Martini: +12 dB)
2. **Temporal density** over iterations or resolution (Piano: +12 dB from 20→600 timesteps)
3. **Restoring opacity_reset** — the single most impactful default config fix (AIST: +1 dB, removes degradation)
4. **Confidence-downweighting** over naive blackout for person masking (Piano: +4.7 dB vs naive mask)
5. **Testing what you actually want** — held-out-VIEW not just held-out-TIME

### What the numbers hide

PSNR is an imperfect proxy. The most important discoveries came from **looking at renders**:
- The 2×-too-wide FOV bug that caused ~2 dB degradation but *looked* like blur, not a unit error
- The floater collapse that *degraded* across training (visible in training curves, not just final metric)
- The novel-view overfit that looked fine from training angles and catastrophic from novel ones

**Render and look at your outputs. Every experiment.**

---

### Resources

| Resource | Link |
|---|---|
| Full code + README | [github.com/Ahmedn1/videosplat](https://github.com/Ahmedn1/videosplat) |
| Detailed experiment write-up | [Findings.md](https://github.com/Ahmedn1/videosplat/blob/main/Findings.md) |
| Live demo (rendered videos + interactive splats) | [videosplat-demo.netlify.app](https://videosplat-demo.netlify.app) |
| Neural 3D Video dataset (N3V) | [facebookresearch/Neural_3D_Video](https://github.com/facebookresearch/Neural_3D_Video) |
| AIST Dance Database + AIST++ calibration | [aistdancedb.ongaaccel.jp](https://aistdancedb.ongaaccel.jp/) |
| Dynamic 3D Gaussians benchmark | [JonathonLuiten/Dynamic3DGaussians](https://github.com/JonathonLuiten/Dynamic3DGaussians) |

**Backbones used:** 4DGaussians · SpacetimeGaussians · Gaussian-Flow · 4D-Rotor-Gaussians · MASt3R